In [ ]:
# Day 14 - LangChain for Easy LLM Applications (Fixed Imports)

!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

# Corrected Imports for newer LangChain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFacePipeline

import torch
from transformers import pipeline


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


/tmp/ipykernel_55337/3857698607.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
# 1. Create Knowledge Base
knowledge = """
Addis Ababa is the capital of Ethiopia and one of the fastest growing cities in Africa.
Ethiopia is known as the cradle of humanity.
Injera is the national dish of Ethiopia made from teff flour.
Lalibela is famous for its rock-hewn churches.
Ethiopia is the birthplace of coffee.
The Ethiopian calendar is 7-8 years behind the Gregorian calendar.
AI and tech startups are rapidly growing in Addis Ababa.
"""

with open("ethiopia_knowledge.txt", "w", encoding="utf-8") as f:
    f.write(knowledge)

# Load documents
loader = TextLoader("ethiopia_knowledge.txt", encoding="utf-8")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)
print(f"Created {len(texts)} document chunks")

In [ ]:
# 2. Create Vector Store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(texts, embeddings)
print("Vector store created!")

In [ ]:
# 3. Load LLM
llm_pipeline = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=300,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)

In [ ]:
# 4. Create RAG Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

# Test
result = qa_chain.invoke({"query": "What is special about Ethiopia?"})
print(result['result'])

In [ ]:
# 5. Simple Chat Function
def ask_question(question):
    result = qa_chain({"query": question})
    print("Question:", question)
    print("Answer:", result['result'])
    return result['result']

ask_question("Tell me about Ethiopian food and culture.")